# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
!git clone https://github.com/muhammadfaseehkhattak/FlyRank-ML-Internship.git

Cloning into 'FlyRank-ML-Internship'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 123 (delta 39), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (123/123), 1.86 MiB | 16.74 MiB/s, done.
Resolving deltas: 100% (39/39), done.


In [5]:
import pandas as pd

df = pd.read_csv(
    "/content/FlyRank-ML-Internship/data/raw/content_refresh_anonymized.csv"
)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [6]:
# Signal 1: Staleness
# Check whether older pages are more likely to have a "down" trend.

staleness_check = (
    df.groupby("freshness_tier", dropna=False)
      .agg(
          n=("content_id", "size"),
          declining=("trend_direction", lambda x: (x == "down").sum())
      )
      .reset_index()
)

staleness_check["decline_rate"] = (
    staleness_check["declining"] / staleness_check["n"] * 100
)

staleness_check = staleness_check.sort_values("decline_rate", ascending=False)

print(staleness_check)

  freshness_tier      n  declining  decline_rate
3         91-180   9171       5604     61.105659
2          31-90    175        103     58.857143
0           0-30  20480      10473     51.137695
1           181+    174         82     47.126437


In [7]:
# Signal 2: Search visibility / volume

volume_check = (
    df.groupby("impression_tier", dropna=False)
      .agg(
          n=("content_id", "size"),
          declining=("trend_direction", lambda x: (x == "down").sum())
      )
      .reset_index()
)

volume_check["decline_rate"] = (
    volume_check["declining"] / volume_check["n"] * 100
)

# Put the tiers in logical order
tier_order = [
    "no_data",
    "none",
    "low",
    "moderate",
    "good",
    "excellent"
]

volume_check["tier_order"] = volume_check["impression_tier"].map(
    {tier: i for i, tier in enumerate(tier_order)}
)

volume_check = volume_check.sort_values("tier_order")

print(volume_check[
    ["impression_tier", "n", "declining", "decline_rate"]
])

  impression_tier      n  declining  decline_rate
2             low  11248       5106     45.394737
3        moderate  10469       6435     61.467189
1            good   7205       4223     58.612075
0       excellent   1078        498     46.196660


### Baseline rule

Prioritize pages that have not been updated for at least 91 days and have at least 300 search impressions over the 90-day window. Among eligible pages, higher-impression pages receive a higher score and are ranked first for review.

### Reason code

- `stale_visible` — the page is stale and still has meaningful search visibility.

### Action label

- `review_refresh` — review the page for a possible content refresh.

### Why this rule

The signal checks showed that staleness and search visibility were both MIXED rather than clean predictors of decline. Therefore, this baseline does not claim that these signals predict decline. Instead, it uses them as transparent decision-support signals: stale pages with meaningful visibility are prioritized because they have both a potential refresh need and measurable search exposure.

The rule uses only current snapshot information and does not use future-window or label-derived fields.

**Staleness verdict — MIXED:**

Pages in the 91–180 day freshness bucket had a higher observed decline rate than recently updated pages, suggesting staleness may be a useful directional signal. However, the 181+ bucket had a lower decline rate and a small sample size, so the relationship is not consistently monotonic.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# Section 2 — Build the ranked baseline queue

baseline = df.copy()

# 1. Define the two conditions
baseline["is_stale"] = (
    baseline["days_since_last_update"] >= 91
).astype(int)

baseline["is_visible"] = (
    baseline["impressions_90d"] >= 300
).astype(int)

# 2. Score:
# Only pages satisfying BOTH conditions receive a score.
# Higher impressions = higher priority.
baseline["score"] = (
    baseline["is_stale"]
    * baseline["is_visible"]
    * baseline["impressions_90d"]
)

# 3. Reason code
baseline["reason_code"] = "not_selected"

baseline.loc[
    (baseline["is_stale"] == 1) &
    (baseline["is_visible"] == 1),
    "reason_code"
] = "stale_visible"

# 4. Action label
baseline["action"] = "no_action"

baseline.loc[
    baseline["reason_code"] == "stale_visible",
    "action"
] = "review_refresh"

# 5. Rank highest score first
baseline = baseline.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

baseline["rank"] = range(1, len(baseline) + 1)

# 6. Show basic results
selected = baseline["reason_code"].eq("stale_visible")

print("Total pages:", len(baseline))
print("Selected for review:", selected.sum())
print("Not selected:", (~selected).sum())

print("\nTop 10:")
print(
    baseline.loc[
        :,
        [
            "rank",
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "score",
            "reason_code",
            "action"
        ]
    ].head(10)
)

Total pages: 30000
Selected for review: 7234
Not selected: 22766

Top 10:
   rank            content_id  days_since_last_update  impressions_90d  \
0     1  content_5fe46e04994d                     104           517715   
1     2  content_2dba2b1f9536                     104           443434   
2     3  content_2c2606c5d176                     104           347399   
3     4  content_cb112fce36be                     104           309910   
4     5  content_9532f197bbc8                     104           309192   
5     6  content_36ff89c8214e                     104           295097   
6     7  content_b28d1efd668f                     104           286608   
7     8  content_813e88069237                     104           233561   
8     9  content_c21024970297                     104           211366   
9    10  content_c8e9d6ab9013                     104           208678   

    score    reason_code          action  
0  517715  stale_visible  review_refresh  
1  443434  stale_visible 

In [11]:
# Save the ranked baseline queue

output_path = "/content/FlyRank-ML-Internship/work/outputs/baseline_action_score.csv"

baseline.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows written:", len(baseline))

Saved: /content/FlyRank-ML-Internship/work/outputs/baseline_action_score.csv
Rows written: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
# Section 3 — Top-20 review

top20 = baseline[
    [
        "rank",
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "score",
        "reason_code",
        "action"
    ]
].head(20)

print(top20.to_string(index=False))

 rank           content_id  days_since_last_update  impressions_90d  score   reason_code         action
    1 content_5fe46e04994d                     104           517715 517715 stale_visible review_refresh
    2 content_2dba2b1f9536                     104           443434 443434 stale_visible review_refresh
    3 content_2c2606c5d176                     104           347399 347399 stale_visible review_refresh
    4 content_cb112fce36be                     104           309910 309910 stale_visible review_refresh
    5 content_9532f197bbc8                     104           309192 309192 stale_visible review_refresh
    6 content_36ff89c8214e                     104           295097 295097 stale_visible review_refresh
    7 content_b28d1efd668f                     104           286608 286608 stale_visible review_refresh
    8 content_813e88069237                     104           233561 233561 stale_visible review_refresh
    9 content_c21024970297                     104           211

In [14]:
top20_details = baseline[
    [
        "rank",
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "content_type",
        "search_volume",
        "score",
        "reason_code",
        "action"
    ]
].head(20)

print(top20_details.to_string(index=False))

 rank           content_id  days_since_last_update  impressions_90d  clicks_90d  ctr  avg_position    content_type  search_volume  score   reason_code         action
    1 content_5fe46e04994d                     104           517715         741 0.14           4.2 keyword article         1900.0 517715 stale_visible review_refresh
    2 content_2dba2b1f9536                     104           443434         910 0.21          27.9 keyword article            0.0 443434 stale_visible review_refresh
    3 content_2c2606c5d176                     104           347399        1854 0.53           4.2 keyword article          590.0 347399 stale_visible review_refresh
    4 content_cb112fce36be                     104           309910         492 0.16           5.6 keyword article           70.0 309910 stale_visible review_refresh
    5 content_9532f197bbc8                     104           309192        2689 0.87           2.0 keyword article           10.0 309192 stale_visible review_refresh
    

In [17]:
# Section 3 — Top-20 review
# Create page-specific review notes based on observable signals.

def review_row(row):
    # Strong candidate: high visibility + poor position
    if row["impressions_90d"] >= 300000 and row["avg_position"] > 20:
        confidence = "High"
        wrong = "The page may have a reason for its low position that a refresh cannot fix."

    # Strong candidate: very high visibility + zero clicks
    elif row["impressions_90d"] >= 200000 and row["clicks_90d"] == 0:
        confidence = "High"
        wrong = "Zero clicks may reflect measurement or SERP conditions rather than outdated content."

    # Strong candidate: good position but unusually low CTR
    elif row["avg_position"] <= 10 and row["ctr"] < 0.2:
        confidence = "High"
        wrong = "The low CTR may come from search-result presentation rather than content quality."

    # Good visibility and reasonable position
    elif row["avg_position"] <= 10:
        confidence = "Moderate"
        wrong = "The page already ranks well, so a refresh may not produce meaningful improvement."

    # Weak position but lower visibility
    elif row["avg_position"] > 20:
        confidence = "Moderate"
        wrong = "The page may have limited opportunity despite being visible in search."

    else:
        confidence = "Moderate"
        wrong = "The page may not need a refresh even though it is stale and visible."

    return pd.Series([confidence, wrong])


top20_review = top20_details.copy()

top20_review[
    ["confidence_note", "what_would_make_it_wrong"]
] = top20_review.apply(review_row, axis=1)

print(top20_review.to_string(index=False))

 rank           content_id  days_since_last_update  impressions_90d  clicks_90d  ctr  avg_position    content_type  search_volume  score   reason_code         action confidence_note                                                             what_would_make_it_wrong
    1 content_5fe46e04994d                     104           517715         741 0.14           4.2 keyword article         1900.0 517715 stale_visible review_refresh            High    The low CTR may come from search-result presentation rather than content quality.
    2 content_2dba2b1f9536                     104           443434         910 0.21          27.9 keyword article            0.0 443434 stale_visible review_refresh            High           The page may have a reason for its low position that a refresh cannot fix.
    3 content_2c2606c5d176                     104           347399        1854 0.53           4.2 keyword article          590.0 347399 stale_visible review_refresh        Moderate    The page alrea

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [18]:
# Section 4 — Weak picks + leakage check

print("WEAK PICKS")
print("=" * 50)

weak_picks = top20_review[
    (top20_review["avg_position"] <= 10) &
    (top20_review["ctr"] >= 0.2)
]

print(weak_picks[
    ["rank", "content_id", "impressions_90d",
     "ctr", "avg_position"]
].to_string(index=False))

print("\nLEAKAGE CHECK")
print("=" * 50)

rule_features = ["days_since_last_update", "impressions_90d"]

for col in rule_features:
    print(f"Used in rule: {col}")

leakage_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

for col in leakage_columns:
    print(f"NOT used: {col}")

print("\nResult: Rule uses current/stable page signals only.")
print("No decline label, trend direction, or future-window information used.")
print("No product flags used.")

WEAK PICKS
 rank           content_id  impressions_90d  ctr  avg_position
    3 content_2c2606c5d176           347399 0.53           4.2
    5 content_9532f197bbc8           309192 0.87           2.0
    9 content_c21024970297           211366 0.41           5.1
   12 content_d17681677e69           201584 0.24           5.8
   15 content_3d94572c3a35           190623 0.24           4.3
   16 content_01908772c6db           187893 0.45           4.0
   20 content_bb5bd5f771dc           176296 0.23           4.3

LEAKAGE CHECK
Used in rule: days_since_last_update
Used in rule: impressions_90d
NOT used: trend_direction
NOT used: trend_pct
NOT used: is_declining_label

Result: Rule uses current/stable page signals only.
No decline label, trend direction, or future-window information used.
No product flags used.


The baseline produced some weak/questionable picks, especially pages that already had good search positions and reasonable CTR. This shows that the simple rule of prioritizing stale, visible pages is useful for ranking but is not perfect.

The leakage check confirmed that the rule used only days_since_last_update and impressions_90d. It did not use trend_direction, trend_pct, is_declining_label, future-window information, or product flags. Therefore, the baseline remains a fair, non-leaking rule that can be compared against a future ML model.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.